In [15]:
import pandas as pd
import numpy as np
import math
from collections import Counter

In [ ]:
df = pd.read_csv("AAPL.csv")

feature_cols = ['Open', 'High', 'Low', 'Close', 'Volume']
df_market = df[feature_cols].copy()
df_market['Target'] = (df_market['Close'].shift(-1) > df_market['Close']).astype(int)
df_data = df_market.iloc[:-1].reset_index(drop=True)

split_idx = int(len(df_data) * 0.67)
train_df = df_data.iloc[:split_idx].reset_index(drop=True)
test_df = df_data.iloc[split_idx:].reset_index(drop=True)

print(f"Total dataset shape: {df_data.shape}")
print(f"Training rows: {len(train_df)} Test rows: {len(test_df)}")
train_df.head()

Total dataset shape: (1592, 6)
Training split rows: 1066 | Verification split rows: 526


,Open,High,Low,Close,Volume,Target
0,374.57,381.80,354.2400,372.50,44059900.0,1
1,367.86,379.82,360.3000,378.25,28088200.0,0
2,373.33,384.78,371.8045,377.37,29020800.0,0
3,375.78,377.74,368.4890,369.80,19123500.0,1
4,379.09,388.81,378.2100,388.81,15804100.0,1


In [17]:
def Entropy(input):
    if not input:
        return 0.0
    list_size = len(input)
    counts = Counter(input)
    entropy = 0.0
    for count in counts.values():
        p = count/list_size
        if p > 0:
            entropy -= p * math.log2(p)
    return entropy

def IG(parent,left,right):
    p_entropy = Entropy(parent)
    p_size = len(parent)
    
    w_left = len(left) / p_size
    w_right = len(right) / p_size
    
    c_entropy = (w_left * Entropy(left)) + (w_right * Entropy(right))
    return p_entropy - c_entropy
    

In [ ]:
def find_best_split(data, target_col='Target'):
    best_gain = -1.0
    best_feature = None
    best_threshold = None
    
    parent_y = data[target_col].tolist()
    feature_cols = [c for c in data.columns if c != target_col]
    
    for col in feature_cols:
        col_data = data[col]
        sorted_vals = np.sort(col_data.unique())
        if len(sorted_vals) < 2:
            continue
        
        midpoints = [(sorted_vals[i] + sorted_vals[i+1]) / 2.0 for i in range(len(sorted_vals) - 1)]
        
        for mid in midpoints:
            left_mask = data[col] <= mid
            left_y = data.loc[left_mask, target_col].tolist()
            right_y = data.loc[~left_mask, target_col].tolist()
            
            gain = IG(parent_y, left_y, right_y)
            if gain > best_gain:
                best_gain = gain
                best_feature = col
                best_threshold = mid
                
    return best_feature, best_threshold, best_gain

In [ ]:
def build_tree(data, target_col='Target', depth=0, max_depth=4, min_samples_split=2):
    target_list = data[target_col].tolist()
    num_samples = len(data)
    unique_classes = set(target_list)
    
    majority_class = Counter(target_list).most_common(1)[0][0] if target_list else 0
    
    if len(unique_classes) == 1 or depth >= max_depth or num_samples < min_samples_split:
        return {'is_leaf': True, 'value': majority_class}
    
    feature, threshold, gain = find_best_split(data, target_col)
    
    if gain <= 0 or feature is None:
        return {'is_leaf': True, 'value': majority_class}
    
    left_df = data[data[feature] <= threshold]
    right_df = data[data[feature] > threshold]
    
    return {
        'is_leaf': False,
        'feature': feature,
        'threshold': threshold,
        'left': build_tree(left_df, target_col, depth + 1, max_depth, min_samples_split),
        'right': build_tree(right_df, target_col, depth + 1, max_depth, min_samples_split)
    }

financial_tree = build_tree(train_df, max_depth=4, min_samples_split=2)

In [32]:
def predict_sample(node,sample_row):
    if node['is_leaf']:
        return node['value']
    
    val = sample_row[node['feature']]
    next_branch = node['left'] if val <= node['threshold'] else node['right']
    
    return predict_sample(next_branch, sample_row)

predictions = [predict_sample(financial_tree, row) for _, row in test_df.iterrows()]
actual_targets = test_df['Target'].tolist()

correct = sum(p == a for p, a in zip(predictions, actual_targets))
accuracy = (correct / len(actual_targets)) * 100

evaluation_results = pd.DataFrame({
    'Actual Direction': actual_targets,
    'Predicted Direction': predictions,
    'Correct Alignment': [p == a for p, a in zip(predictions, actual_targets)]
})

print(f"Accuracy: {accuracy:.2f}%\n")
evaluation_results

Accuracy: 48.86%



,Actual Direction,Predicted Direction,Correct Alignment
0,0,1,False
1,1,1,True
2,0,1,False
3,0,1,False
4,0,1,False
...,...,...,...
521,1,0,False
522,0,0,True
523,0,0,True
524,1,0,False
